# Prompt Chaining for Iterative Story Generation

A bootcamp exercise demonstrating how multiple LLM calls can be connected into a controlled generation workflow. The notebook builds a science-fiction story in stages rather than asking the model to generate the entire result in one call.

### Pipeline

```text
Premise → Outline → Opening → Continuation → Final Story
```

The focus is prompt chaining, state passing, iterative generation, and explicit stopping conditions.


## Learning outcomes

This notebook demonstrates how to:

- pass the output of one prompt into the next.
- combine persona, context, constraints, and output instructions.
- generate long-form content across multiple model calls.
- preserve story state between calls.
- stop an iterative generation loop with both a completion marker and a maximum-step limit.


## Implementation notes

- The OpenAI API key is entered at runtime with `getpass` and is not stored in the notebook.
- Prompt templates use placeholders filled later with `.format()`.
- The workflow uses `IAMDONE` as a completion marker.
- A maximum number of continuations prevents an unbounded generation loop.


## 1. Setup


In [8]:
!pip install -q -U openai

from getpass import getpass
from openai import OpenAI

api_key = getpass("Enter your OpenAI API key: ")
client = OpenAI(api_key=api_key)
MODEL = "gpt-5-mini"

print("Setup complete")

Enter your OpenAI API key: ··········
Setup complete


## 2. Shared writing instructions

A reusable persona and writing guidelines keep the tone and constraints consistent across the full prompt chain.


In [ ]:
persona = "You are a creative science-fiction author writing for a general audience."

guidelines = """
Write vivid scenes with sensory details.
Develop the characters' goals and conflicts.
Do not summarize the story too quickly.
Continue naturally from the existing draft.
"""

print(persona)

You are a creative science-fiction author writing for a general audience.


## 3. Build the prompt chain

The workflow uses separate templates for the premise, outline, opening, and continuation stages. Placeholders such as `{premise}`, `{outline}`, and `{story_text}` allow generated content from earlier stages to become context for later calls.


In [9]:
premise_prompt = f"""
{persona}

Write one exciting sentence for a science-fiction story about a lost city on Mars.
"""

# TODO 1: Write outline_prompt using the premise placeholder.
# Ask the model for 5-7 major plot points.
outline_prompt = f"""
{persona}

Here is the premise of the story:

{{premise}}

Create an outline for this science-fiction story with 5-7 major plot points.
Make sure the plot develops logically from the premise.
"""

# TODO 2: Write starting_prompt using the premise and outline placeholders.
# Ask for 500-800 words and introduce at least one important character.
starting_prompt = f"""
{persona}

Here is the premise:

{{premise}}

Here is the story outline:

{{outline}}

Write the beginning of the story in 500-800 words.
Introduce at least one important character and follow the premise and outline.
Do not finish the story yet.
"""

# TODO 3: Write continuation_prompt using premise, outline, and story_text.
# Ask the model to continue the story and write IAMDONE when completely finished.
continuation_prompt = f"""
{persona}

Here is the premise:

{{premise}}

Here is the story outline:

{{outline}}

Here is the story written so far:

{{story_text}}

Continue the story from where it stopped.
Follow the premise and outline and avoid repeating previous events.

When the story is completely finished, write IAMDONE at the end.
"""

## 4. Generate the premise

The first model call creates a concise premise that becomes the foundation for the rest of the chain.


In [11]:
response = client.responses.create(
    model=MODEL,
    input=premise_prompt
)

premise = response.output_text

print("Premise:")
print(premise)

Premise:
The sand gave way and the lost city of Mars rose around us—ancient metal spires humming to life as holographic constellations unfolded overhead, spelling out a welcome none of us could read and a warning none of us could ignore.


In [12]:
# TODO 4: Generate the outline with outline_prompt.format(premise=premise).
# Save the model output in outline and print it.
response = client.responses.create(
    model=MODEL,
    input=outline_prompt.format(premise=premise)
)

outline = response.output_text

print("Outline:")
print(outline)

Outline:
1) The Sand Gives Way — Discovery and Awakening
- A mixed exploration team (archaeologist protagonist, a corporate representative, a field engineer, and a colonial security officer) is mapping a dunescape when the ground collapses and reveals a sunken avenue of metal spires.  
- As they descend, the spires hum, plates shift, and holographic constellations ripple into the Martian sky, assembling a luminous script: a ceremonial welcome in an unknown syntax that folds into a second, unmistakable warning line.  
- The scene sets the tone: wonder and promise on one hand, and a mechanical, ancient menace on the other; the team radios their find and a scramble of political and commercial interest begins.

2) First Contact — Exploration, Translation, and Friction
- The team begins surface surveys and pulls fragments of the city’s interface into their mobile lab. The holograms respond to attention and mirror human gestures, complicating simple testing.  
- Tension rises as external act

## 5. Generate the outline

The generated premise is inserted into the outline prompt so the next model call can plan the major plot points.


In [13]:
response = client.responses.create(
    model=MODEL,
    input=outline_prompt.format(premise=premise),
)

outline = response.output_text.strip()
print("Outline:")
print(outline)

Outline:
1) Inciting discovery — the city wakes
- A routine survey mission to a buried basin on Mars collapses the surface; ancient metal spires rise, rotating into place as dust and ice fall away. The expedition—scientists, a linguist-protagonist (Mara), an engineer, a conflicted corporate overseer, a veteran pilot, and an exobiologist—watch holographic constellations bloom overhead. The images read like both a greeting and a threat: patterns none can read but whose cadence and emphasis create an unmistakable urgency. The team’s conflicting priorities (science, profit, safety) are immediately set against an urgent, unknown risk.

2) First contact and the puzzle of meaning
- The team rigs interfaces, samples energy signatures and holographic encodings; their shipboard AIs begin pattern-matching. Mara discovers the ‘language’ is not phonetic but astrophysical—glyphs map to star charts, orbital mechanics and time cycles. Archaeological evidence in exposed chambers (ruined automatons, sco

## 6. Generate the opening

The premise and outline are passed together to the model to create the opening section and introduce the story's important characters.


In [14]:
# TODO 5: Generate the opening with starting_prompt.format(...).
# Save the model output in starting_draft and print it.
response = client.responses.create(
    model=MODEL,
    input=starting_prompt.format(
        premise=premise,
        outline=outline
    )
)
starting_draft = response.output_text

print("Opening:")
print(starting_draft)

Opening:
The sand gave way beneath the riggers' cleats and the world folded open.

One moment the basin was a flat of wind-etched ochre and frost; the next the ground yawned and a lattice of old metal, black as spent lightning, rolled up out of the dust. Spires uncoiled like the ribs of a sleeping beast, pivoting into place with oiled precision, joints that had not moved in millennia whispering to life. Ice beads fell from their flanges and clattered against the hull of the survey skiff. A thin, low hum filled the air, a frequency that made Mara's fillings ache and the hair on her forearm stand up.

Above that metal forest something older still unfurled: filaments of light braided themselves into arcs and points, constellations ghosting over the ruins in a kind of living map. They were not the stars the humans knew—no Orion, no familiar polarisms—but shapes that shifted and nested, like a finger tracing orbital paths across a tabletop. A chorus of pulses accompanied them, not sounds in

## 7. Generate the first continuation

The first continuation is produced separately so the state-passing pattern can be inspected before moving into the loop.


In [15]:
draft = starting_draft

# TODO 6: Call the model with continuation_prompt.format(...).
# Use premise, outline, and story_text=draft.
continuation = client.responses.create(
    model=MODEL,
    input=continuation_prompt.format(
        premise=premise,
        outline=outline,
        story_text=draft
    )
).output_text

print("Continuation:")
print(continuation)

Continuation:
Alden's uplink answered with a human heartbeat of confirmation and then, in the small theater of the skiff's comm loop, other heartbeats—voices from headquarters, then a ping from a comm-wafer that hadn't existed a week ago, a corporate envoy's call to launch a secure hold. The badge at Alden's throat flared with authority, and his smile sharpened.

"Central wants a live feed," he said. "They want eyes. They'll want control. We stay in the loop, we keep the rights."

Kade put a steady hand on the spire's flange and looked at the rising constellations the way a man reads the sea. "If you hit record, you can't rewind what it sends."

"Then don't let it finish," Élodie said. She had moved from the frost-scoured metal to a shattered dome where a crust of organic residue glittered like dried pollen. She'd photographed, sampled, and catalogued on autopilot, eyes wide and furious. "We've got microbes here with non-Earth signatures. Enzymes tuned to silica, exotoxin scaffolding. 

## 8. Continue iteratively

Each new continuation is appended to the current draft and passed back into the next prompt. The loop stops when the model emits `IAMDONE` or when the maximum number of continuation calls is reached.


In [16]:
# TODO 7: Add the first continuation to draft.
draft += "\n" + continuation

# TODO 8: Set a maximum number of additional calls.
MAX_CONTINUATIONS = 3

for _ in range(MAX_CONTINUATIONS):
    # TODO 9: Stop if IAMDONE appears in continuation.
    if "IAMDONE" in continuation:
        break

    # TODO 10: Request the next continuation.
    continuation = client.responses.create(
        model=MODEL,
        input=continuation_prompt.format(
            premise=premise,
            outline=outline,
            story_text=draft
        )
    ).output_text

    # TODO 11: Append the new continuation to draft.
    draft += "\n" + continuation

# TODO 12: Remove IAMDONE and save the cleaned story in final.
final = draft.replace("IAMDONE", "").strip()

print(final)

The sand gave way beneath the riggers' cleats and the world folded open.

One moment the basin was a flat of wind-etched ochre and frost; the next the ground yawned and a lattice of old metal, black as spent lightning, rolled up out of the dust. Spires uncoiled like the ribs of a sleeping beast, pivoting into place with oiled precision, joints that had not moved in millennia whispering to life. Ice beads fell from their flanges and clattered against the hull of the survey skiff. A thin, low hum filled the air, a frequency that made Mara's fillings ache and the hair on her forearm stand up.

Above that metal forest something older still unfurled: filaments of light braided themselves into arcs and points, constellations ghosting over the ruins in a kind of living map. They were not the stars the humans knew—no Orion, no familiar polarisms—but shapes that shifted and nested, like a finger tracing orbital paths across a tabletop. A chorus of pulses accompanied them, not sounds in the air 

## 9. Inspect the final result

The completed story is printed, its length is measured, and the completion marker is checked.


In [17]:
# TODO 13: Count the words in final.
word_count = len(final.split())
print(f"Final story word count: {word_count}")

# Extra check added for this student task: verify that the completion marker is removed.
print("Completion marker removed:", "IAMDONE" not in final)

Final story word count: 5324
Completion marker removed: True


## What this notebook demonstrates

- Prompt chaining across multiple LLM calls
- Reusing generated outputs as downstream context
- Iterative generation with accumulated state
- Explicit stopping conditions and bounded loops
- Separation of prompt templates from generated content
